# Pipeline de Consolidación — Sala de Detenidos
## Versión 2.0 — Construida sobre análisis real de los archivos

### Lo que descubrió el análisis de los archivos reales

| Hallazgo | Impacto en v1 | Solución v2 |
|---|---|---|
| Archivos tienen **96 columnas reales** | `iloc[:, :60]` descartaba todo el bloque ESTACIONES | Se leen todas las columnas |
| Archivos tienen **1.079 filas de datos** | `pd.read_excel` solo veía 469 por merged cells gigantes | Se usa `openpyxl` puro |
| Encabezado está en **filas 4, 5, 6** (índices 3,4,5) | La búsqueda del ancla funcionaba por azar | Posición fija confirmada |
| **Merged cells** cubren RG y UNIDAD por decenas de filas | NaN masivo en columnas de ID | Expansión previa con openpyxl |
| Cols con merged cells en datos (36, 37) | Valores de URI aparecían como NaN | Expansión total antes de leer |
| **3 niveles** de encabezado (F4=bloque, F5=subgrupo, F6=campo) | Nombres mal construidos | Propagación correcta nivel a nivel |

### Estructura confirmada del archivo
```
Fila 1-3  : Metadata / vacío
Fila 4    : Nivel 1 — 'RG', 'UNIDAD', 'UBICACIÓN SALAS', 'TOTAL...', 'DETENIDOS POLICIALES', 'DETENIDOS URI'
Fila 5    : Nivel 2 — 'CONDICIÓN', 'GENERO', 'PARTE PERSONAL', 'ALIMENTACIÓN'...
Fila 6    : Nivel 3 — 'N° CONDENADOS', 'M', 'F', 'LGBTI', 'OF', 'M/E'...
Fila 7+   : Datos reales (~1.079 filas por archivo)
Fila 1086 : TOTAL GENERAL  ← corte
Fila 1087+: Verificaciones y notas al pie
```

---
## ⚙️ CELDA 1 — Configuración (única celda que debes editar)

In [1]:
import os

# ─── AJUSTA SOLO ESTA RUTA ────────────────────────────────────────────────────
RUTA_BASE = r'C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases'
# ─────────────────────────────────────────────────────────────────────────────

RUTA_RESULTADOS = os.path.join(RUTA_BASE, 'Resultados')

# Filas del encabezado (índices 0-based dentro del DataFrame de openpyxl)
# Confirmado por análisis: fila Excel 4 = índice 3, fila Excel 5 = índice 4, fila Excel 6 = índice 5
IDX_NIVEL1 = 3   # Bloque principal: 'DETENIDOS EN INSTALACIONES POLICIALES', 'URI'
IDX_NIVEL2 = 4   # Subgrupo: 'CONDICIÓN', 'GENERO', 'PARTE PERSONAL'...
IDX_NIVEL3 = 5   # Campo: 'N° CONDENADOS', 'M', 'F', 'LGBTI', 'OF'...
IDX_DATOS  = 6   # Primera fila de datos reales (fila Excel 7)

# Palabras que marcan el fin de los datos
PALABRAS_CORTE = {'TOTAL GENERAL', 'VERIFICACIÓN', 'DETENIDOS ENFERMOS'}

# Columnas de identificación (no se convierten a numérico)
COLS_TEXTO = {'RG', 'UNIDAD', 'UBICACIÓN SALAS', 'ARCHIVO_ORIGEN', 'FECHA_REPORTE'}

# Diccionario de homologación de vocabulario entre operadores
HOMOLOGACION = {
    'RETENIDOS'   : 'DETENIDOS',
    'RETENIDAS'   : 'DETENIDAS',
    'SINDICADOS'  : 'IMPUTADOS',
    'SINDICADO'   : 'IMPUTADO',
    'UBICIACIÓN'  : 'UBICACIÓN',
    'CONDICION_'  : 'CONDICIÓN_',
    'CONDICION'   : 'CONDICIÓN',
    'CONDICION '  : 'CONDICIÓN ',
    'EXTRAJEROS'  : 'EXTRANJEROS',
    'VENEZOLANOS' : '(VENEZOLANOS)',
    'ALIMENTACION': 'ALIMENTACIÓN',
}

# Validar ruta
if not os.path.isdir(RUTA_BASE):
    raise FileNotFoundError(
        f"❌ No se encontró: '{RUTA_BASE}'\n"
        "Verifica que OneDrive esté sincronizado o ajusta RUTA_BASE."
    )
os.makedirs(RUTA_RESULTADOS, exist_ok=True)
print(f"✅ Ruta base     : {RUTA_BASE}")
print(f"✅ Resultados en : {RUTA_RESULTADOS}")

✅ Ruta base     : C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases
✅ Resultados en : C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases\Resultados


---
## 📦 CELDA 2 — Importaciones

In [2]:
import re
import warnings
from datetime import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl

pd.set_option('future.no_silent_downcasting', True)
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

print(f"pandas   : {pd.__version__}")
print(f"openpyxl : {openpyxl.__version__}")
print(f"numpy    : {np.__version__}")
print(f"Ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

pandas   : 2.3.1
openpyxl : 3.1.5
numpy    : 2.3.1
Ejecución: 2026-03-31 18:02


---
## 🛠️ CELDA 3 — Funciones auxiliares

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER 1: Limpiar valor de celda
# ─────────────────────────────────────────────────────────────────────────────
def limpiar(valor) -> str:
    """Convierte un valor de celda a string limpio y en mayúsculas."""
    if valor is None:
        return ''
    s = str(valor).strip()
    if s.lower() in ('none', 'nan', ''):
        return ''
    # Normalizar saltos de línea dentro de celdas (frecuentes en % HACINAMIENTO)
    s = re.sub(r'[\r\n]+', ' ', s)
    # Normalizar espacios múltiples
    s = re.sub(r'\s+', ' ', s).strip()
    return s.upper()


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 2: Deduplicar lista de nombres (bug corregido vs v1)
# ─────────────────────────────────────────────────────────────────────────────
def deduplicar(nombres: list) -> list:
    """
    Garantiza nombres únicos con sufijos incrementales.
    Resultado para ['X','X','X','Y','Y'] → ['X','X_1','X_2','Y','Y_1']

    Corrección del bug de v1 donde el tercer duplicado también recibía _1.
    """
    conteo    = defaultdict(int)
    resultado = []
    for nombre in nombres:
        if conteo[nombre] == 0:
            resultado.append(nombre)
        else:
            resultado.append(f"{nombre}_{conteo[nombre]}")
        conteo[nombre] += 1
    return resultado


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 3: Homologar nombre de columna
# ─────────────────────────────────────────────────────────────────────────────
def homologar(nombre: str) -> str:
    """Aplica el diccionario de homologación a un nombre de columna."""
    for original, estandar in HOMOLOGACION.items():
        nombre = nombre.replace(original, estandar)
    return nombre


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 4: Extraer fecha del nombre de archivo
# ─────────────────────────────────────────────────────────────────────────────
def extraer_fecha(nombre_archivo: str) -> str:
    """
    Intenta extraer la fecha del nombre del archivo.
    Soporta formatos: DDMMYYYY o DD-MM-YYYY presentes en el nombre.
    Ejemplos:
      'SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx' → '2024-01-23'
      'SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx' → '2025-02-12'
    """
    # Buscar patrón de 8 dígitos consecutivos (DDMMYYYY)
    match = re.search(r'(\d{2})(\d{2})(\d{4})', nombre_archivo)
    if match:
        dd, mm, yyyy = match.group(1), match.group(2), match.group(3)
        try:
            fecha = datetime(int(yyyy), int(mm), int(dd))
            return fecha.strftime('%Y-%m-%d')
        except ValueError:
            pass
    return ''


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 5: Detectar operador desde el nombre del archivo
# ─────────────────────────────────────────────────────────────────────────────
def extraer_operador(nombre_archivo: str) -> str:
    """
    Extrae el apellido del operador que diligencia el reporte.
    Busca el patrón 'IT. APELLIDO', 'PT. APELLIDO', 'SI. APELLIDO', etc.
    """
    match = re.search(
        r'(?:IT[_\.]+|PT[_\.]+|SI[_\.]+|IJ[_\.]+)\s*([A-ZÁÉÍÓÚÑ]+)',
        nombre_archivo.upper()
    )
    return match.group(1) if match else 'DESCONOCIDO'


print("✅ Funciones auxiliares cargadas.")
# Pruebas rápidas
assert deduplicar(['X','X','X','Y']) == ['X','X_1','X_2','Y'], "Bug en deduplicar"
assert extraer_fecha('SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx') == '2024-01-23'
assert extraer_fecha('SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx') == '2025-02-12'
assert extraer_operador('SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx') == 'AYALA'
print("✅ Tests de helpers pasados.")


✅ Funciones auxiliares cargadas.
✅ Tests de helpers pasados.


---
## 🔧 CELDA 4 — Función principal de lectura y transformación

In [4]:
def transformar_archivo(ruta_archivo: str, verbose: bool = True) -> pd.DataFrame | None:
    """
    Lee un archivo Excel de 'Sala de Detenidos', expande todas las celdas
    combinadas, construye nombres de columna desde los 3 niveles de encabezado
    y retorna un DataFrame limpio con los datos reales.

    Parámetros
    ----------
    ruta_archivo : str
        Ruta completa al archivo .xlsx
    verbose : bool
        Si True imprime detalles del proceso (útil para depuración)

    Retorna
    -------
    pd.DataFrame con los datos limpios, o None si el archivo falla.

    Notas de implementación
    -----------------------
    - Se usa openpyxl PURO (no pd.read_excel) porque pd.read_excel no expande
      las celdas combinadas, devolviendo NaN en las celdas secundarias de cada
      merged range. Con archivos que tienen merged cells de 100+ filas en las
      columnas RG y UNIDAD, esto resulta en pérdida masiva de datos.
    - pd.read_excel también truncaba la lectura a ~469 filas cuando el archivo
      real tiene 1.079 filas de datos, por el mismo problema de merged cells.
    """
    nombre_archivo = os.path.basename(ruta_archivo)

    try:
        # ══════════════════════════════════════════════════════════════════════
        # PASO 1: Cargar con openpyxl y expandir TODAS las celdas combinadas
        # ══════════════════════════════════════════════════════════════════════
        wb = openpyxl.load_workbook(ruta_archivo, data_only=True)
        ws = wb.active

        # Copiar rangos antes de iterar (la colección muta al hacer unmerge)
        rangos = list(ws.merged_cells.ranges)
        n_merged = len(rangos)

        for rango in rangos:
            # El valor real siempre está en la celda ancla (esquina sup-izq)
            valor_ancla = ws.cell(row=rango.min_row, column=rango.min_col).value
            ws.unmerge_cells(str(rango))
            # Propagar a todas las celdas que cubrían la celda combinada
            for fila in ws.iter_rows(
                min_row=rango.min_row, max_row=rango.max_row,
                min_col=rango.min_col, max_col=rango.max_col
            ):
                for celda in fila:
                    celda.value = valor_ancla

        # Volcar la hoja a DataFrame sin asumir encabezados
        df_raw = pd.DataFrame(ws.values)
        wb.close()

        if verbose:
            print(f"   Merged cells expandidas  : {n_merged}")
            print(f"   Shape crudo (filas×cols) : {df_raw.shape}")

        # ══════════════════════════════════════════════════════════════════════
        # PASO 2: Construir nombres de columna desde los 3 niveles de encabezado
        # Confirmado por análisis: filas Excel 4,5,6 → índices 3,4,5
        # ══════════════════════════════════════════════════════════════════════
        fila_n1 = [limpiar(v) for v in df_raw.iloc[IDX_NIVEL1].tolist()]
        fila_n2 = [limpiar(v) for v in df_raw.iloc[IDX_NIVEL2].tolist()]
        fila_n3 = [limpiar(v) for v in df_raw.iloc[IDX_NIVEL3].tolist()]

        n_cols = df_raw.shape[1]
        nuevos_nombres = []
        n1_actual = ''
        n2_actual = ''

        for i in range(n_cols):
            v1 = fila_n1[i] if i < len(fila_n1) else ''
            v2 = fila_n2[i] if i < len(fila_n2) else ''
            v3 = fila_n3[i] if i < len(fila_n3) else ''

            # Propagar nivel 1 y 2 hacia la derecha
            # (tras expandir merged cells ya están propagados, pero
            #  se mantiene la lógica por si algún archivo no los tiene)
            if v1 and 'UNNAMED' not in v1:
                n1_actual = v1
            if v2 and 'UNNAMED' not in v2:
                n2_actual = v2

            # Las primeras 3 columnas son IDs (RG, UNIDAD, UBICACIÓN)
            # No necesitan prefijo de bloque
            if i < 3:
                nombre = v1 or f'COL_{i}'
            elif i in (3, 4):
                # Columnas de totales generales (nivel 1 ocupa 3 filas)
                nombre = v1 or v2 or f'COL_{i}'
            else:
                # Construir nombre jerárquico: BLOQUE_SUBGRUPO_CAMPO
                partes = [p for p in [n1_actual, n2_actual, v3] if p]
                # Si no hay nivel 3, usar nivel 2 como hoja final
                if not partes:
                    partes = [f'COL_{i}']
                nombre = '_'.join(partes)

            # Limpiar el nombre final
            nombre = re.sub(r'\s+', ' ', nombre).strip('_').strip()
            nuevos_nombres.append(nombre if nombre else f'COL_{i}')

        # Deduplicar nombres (por si dos columnas distintas producen el mismo nombre)
        nuevos_nombres = deduplicar(nuevos_nombres)

        # ══════════════════════════════════════════════════════════════════════
        # PASO 3: Extraer el área de datos reales
        # Datos comienzan en índice IDX_DATOS (fila Excel 7)
        # Datos terminan cuando aparece 'TOTAL GENERAL' en col 0 o 1
        # ══════════════════════════════════════════════════════════════════════
        df_datos = df_raw.iloc[IDX_DATOS:].copy().reset_index(drop=True)
        df_datos.columns = nuevos_nombres

        # Detectar la fila de corte (TOTAL GENERAL u otras palabras de fin)
        fila_corte = None
        col_rg     = df_datos.columns[0]   # 'RG'
        col_unidad = df_datos.columns[1]   # 'UNIDAD'

        for idx in df_datos.index:
            vals_id = [
                limpiar(df_datos.loc[idx, col_rg]),
                limpiar(df_datos.loc[idx, col_unidad])
            ]
            if any(
                any(palabra in val for palabra in PALABRAS_CORTE)
                for val in vals_id
            ):
                fila_corte = idx
                break

        if fila_corte is not None:
            df_datos = df_datos.iloc[:fila_corte].copy()

        if verbose:
            print(f"   Fila de corte encontrada : índice {fila_corte} "
                  f"({'TOTAL GENERAL' if fila_corte else 'no encontrada'})")

        # ══════════════════════════════════════════════════════════════════════
        # PASO 4: Eliminar columnas completamente vacías
        # (columnas con header pero sin datos reales)
        # ══════════════════════════════════════════════════════════════════════
        n_cols_antes = df_datos.shape[1]
        df_datos = df_datos.dropna(axis=1, how='all')
        n_cols_drop = n_cols_antes - df_datos.shape[1]
        if verbose and n_cols_drop > 0:
            print(f"   Columnas vacías eliminadas: {n_cols_drop}")

        # ══════════════════════════════════════════════════════════════════════
        # PASO 5: Eliminar filas completamente vacías ANTES del ffill
        # (orden crítico: v1 hacía ffill primero, llenando filas fantasma)
        # ══════════════════════════════════════════════════════════════════════
        n_antes = len(df_datos)
        df_datos = df_datos.dropna(how='all').reset_index(drop=True)
        if verbose and (n_antes - len(df_datos)) > 0:
            print(f"   Filas vacías eliminadas  : {n_antes - len(df_datos)}")

        # ══════════════════════════════════════════════════════════════════════
        # PASO 6: ffill en columnas de identificación
        # Sólo DESPUÉS del dropna para no propagar a filas fantasma
        # RG y UNIDAD son merged cells verticales → el valor solo está
        # en la primera celda de cada grupo; las demás ya están propagadas
        # por el paso 1, pero se aplica ffill como red de seguridad
        # ══════════════════════════════════════════════════════════════════════
        cols_id = df_datos.columns[:3]  # RG, UNIDAD, UBICACIÓN SALAS
        df_datos[cols_id] = (
            df_datos[cols_id]
            .replace(['', 'None', 'nan', 'NaN', 'NAN', None], pd.NA)
            .ffill()
        )

        # ══════════════════════════════════════════════════════════════════════
        # PASO 7: Filtrar filas de subtotales
        # ══════════════════════════════════════════════════════════════════════
        mask_sub = df_datos[col_unidad].astype(str).str.contains(
            'SUBTOTAL', case=False, na=False
        )
        n_sub = mask_sub.sum()
        df_datos = df_datos[~mask_sub].reset_index(drop=True)
        if verbose and n_sub > 0:
            print(f"   Filas subtotal eliminadas: {n_sub}")

        # ══════════════════════════════════════════════════════════════════════
        # PASO 8: Metadatos de trazabilidad
        # ══════════════════════════════════════════════════════════════════════
        df_datos.insert(0, 'ARCHIVO_ORIGEN', nombre_archivo)
        df_datos.insert(1, 'FECHA_REPORTE',  extraer_fecha(nombre_archivo))
        df_datos.insert(2, 'OPERADOR',        extraer_operador(nombre_archivo))

        if verbose:
            print(f"   ✅ Resultado: {df_datos.shape[0]} filas × {df_datos.shape[1]} columnas")

        return df_datos

    except Exception as e:
        import traceback
        print(f"   ❌ Error: {type(e).__name__}: {e}")
        if verbose:
            traceback.print_exc()
        return None


print("✅ Función transformar_archivo() cargada.")

✅ Función transformar_archivo() cargada.


---
## 🧪 CELDA 5 — Prueba con un archivo antes de procesar todos

In [5]:
# Tomar el primer archivo para validar que todo funcione correctamente
archivos_disponibles = sorted([
    f for f in os.listdir(RUTA_BASE)
    if f.lower().endswith(('.xlsx', '.xls')) and not f.startswith('~$')
])

print(f"📂 Total de archivos encontrados: {len(archivos_disponibles)}")

# Probar con el primero de la lista
archivo_prueba = archivos_disponibles[0]
print(f"\n🧪 Probando con: {archivo_prueba}\n")
print("-" * 60)

df_prueba = transformar_archivo(
    os.path.join(RUTA_BASE, archivo_prueba),
    verbose=True
)

if df_prueba is not None:
    print("-" * 60)
    print("\n📋 Primeras 5 columnas, 5 filas:")
    display(df_prueba.iloc[:5, :8])

    print("\n📋 Columnas generadas:")
    for i, col in enumerate(df_prueba.columns):
        print(f"  [{i:3d}] {col}")

    print(f"\n📊 NaN por columna (solo las que tienen):")
    nan_counts = df_prueba.isnull().sum()
    nan_counts = nan_counts[nan_counts > 0]
    if nan_counts.empty:
        print("  ✅ Sin NaN")
    else:
        for col, cnt in nan_counts.items():
            pct = cnt / len(df_prueba) * 100
            print(f"  [{pct:5.1f}%] {col}")
else:
    print("❌ La prueba falló. Revisar los errores arriba.")

📂 Total de archivos encontrados: 22

🧪 Probando con: SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx

------------------------------------------------------------
   Merged cells expandidas  : 214
   Shape crudo (filas×cols) : (1060, 96)
   Fila de corte encontrada : índice 1034 (TOTAL GENERAL)
   Columnas vacías eliminadas: 10
   Filas subtotal eliminadas: 10
   ✅ Resultado: 1024 filas × 89 columnas
------------------------------------------------------------

📋 Primeras 5 columnas, 5 filas:


,ARCHIVO_ORIGEN,FECHA_REPORTE,OPERADOR,RG,UNIDAD,UBICACIÓN SALAS,TOTAL DE PERSONAS DETENIDAS EN INSTALACIONES POLICIALES Y URI,TOTAL DEL PERSONAL QUE CUSTODIA LOS DETENIDOS EN INSTALACIONES POLICIALES Y URI
0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,2024-01-23,AYALA,REMSA,MEBOG,USAQUÉN,84,24
1,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,2024-01-23,AYALA,REMSA,MEBOG,CHAPINERO,0,0
2,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,2024-01-23,AYALA,REMSA,MEBOG,SANTAFÉ,108,15
3,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,2024-01-23,AYALA,REMSA,MEBOG,SAN CRISTÓBAL,114,15
4,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,2024-01-23,AYALA,REMSA,MEBOG,USME,95,16



📋 Columnas generadas:
  [  0] ARCHIVO_ORIGEN
  [  1] FECHA_REPORTE
  [  2] OPERADOR
  [  3] RG
  [  4] UNIDAD
  [  5] UBICACIÓN SALAS
  [  6] TOTAL DE PERSONAS DETENIDAS EN INSTALACIONES POLICIALES Y URI
  [  7] TOTAL DEL PERSONAL QUE CUSTODIA LOS DETENIDOS EN INSTALACIONES POLICIALES Y URI
  [  8] DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO_CANTIDAD DE SALAS DEPTO O METRO
  [  9] DETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO_CAPACIDAD DE LAS SALAS DEPTO O METRO
  [ 10] DETENIDOS EN INSTALACIONES POLICIALES_IMPUTADOS CON MAS DE 12 MESES_IMPUTADOS CON MAS DE 12 MESES
  [ 11] DETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS_TOTAL DE PERSONAS EN LAS SALAS
  [ 12] DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS
  [ 13] DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° IMPUTADOS
  [ 14] DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_CUANTOS NO TIENEN LA DOCUMENTACIÓN COMPLETA PARA TRASLADO A CENTRO C

---
## 🔄 CELDA 6 — FASE 1: Procesar todos los archivos

In [6]:
print("═" * 65)
print("FASE 1 — EXTRACCIÓN Y TRANSFORMACIÓN INDIVIDUAL")
print("═" * 65)

todos_los_datos  = []
archivos_fallidos = []
resumen_archivos  = []

for archivo in archivos_disponibles:
    ruta_full = os.path.join(RUTA_BASE, archivo)
    print(f"\n📖 {archivo}")

    df_temp = transformar_archivo(ruta_full, verbose=True)

    if df_temp is not None:
        todos_los_datos.append(df_temp)
        resumen_archivos.append({
            'Archivo'       : archivo,
            'Fecha_reporte' : df_temp['FECHA_REPORTE'].iloc[0],
            'Operador'      : df_temp['OPERADOR'].iloc[0],
            'Filas'         : df_temp.shape[0],
            'Columnas'      : df_temp.shape[1],
            'Estado'        : '✅'
        })
    else:
        archivos_fallidos.append(archivo)
        resumen_archivos.append({
            'Archivo'       : archivo,
            'Fecha_reporte' : '',
            'Operador'      : '',
            'Filas'         : 0,
            'Columnas'      : 0,
            'Estado'        : '❌'
        })

print("\n" + "═" * 65)
print(f"✅ Procesados correctamente : {len(todos_los_datos)}/{len(archivos_disponibles)}")
if archivos_fallidos:
    print(f"❌ Con error               : {len(archivos_fallidos)}")
    for af in archivos_fallidos:
        print(f"   - {af}")

print("\n📋 Resumen por archivo:")
df_resumen = pd.DataFrame(resumen_archivos)
display(df_resumen)

═════════════════════════════════════════════════════════════════
FASE 1 — EXTRACCIÓN Y TRANSFORMACIÓN INDIVIDUAL
═════════════════════════════════════════════════════════════════

📖 SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx
   Merged cells expandidas  : 214
   Shape crudo (filas×cols) : (1060, 96)
   Fila de corte encontrada : índice 1034 (TOTAL GENERAL)
   Columnas vacías eliminadas: 10
   Filas subtotal eliminadas: 10
   ✅ Resultado: 1024 filas × 89 columnas

📖 SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx
   Merged cells expandidas  : 240
   Shape crudo (filas×cols) : (1105, 96)
   Fila de corte encontrada : índice 1079 (TOTAL GENERAL)
   Columnas vacías eliminadas: 10
   Filas subtotal eliminadas: 10
   ✅ Resultado: 1069 filas × 89 columnas

📖 SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx
   Merged cells expandidas  : 240
   Shape crudo (filas×cols) : (1105, 96)
   Fila de corte encontrada : índice 1079 (TOTAL GENERAL)
   Columnas vacías eliminadas: 10
   Filas subtotal e

,Archivo,Fecha_reporte,Operador,Filas,Columnas,Estado
0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,2024-01-23,AYALA,1024,89,✅
1,SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx,2025-02-12,AYALA,1069,89,✅
2,SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx,2025-02-23,AYALA,1069,89,✅
3,SALA DE DETENIDOS DÍA - 28022025 IT. AYALA.xlsx,2025-02-28,AYALA,1069,89,✅
4,SALA DE DETENIDOS DÍA - 29092024 IT. AYALA.xlsx,2024-09-29,AYALA,1069,89,✅
5,SALA DE DETENIDOS DÍA - 31082024 IT. AYALA.xlsx,2024-08-31,AYALA,1069,89,✅
6,SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx,2025-03-02,DÁVILA,1069,89,✅
7,SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx,2025-03-10,DÁVILA,1064,70,✅
8,SALA DE DETENIDOS DÍA 12032025 PT. DÁVILA.xlsx,2025-03-12,DÁVILA,1072,63,✅
9,SALA DE DETENIDOS DÍA 17032025 IT. ACOSTA.xlsx,2025-03-17,ACOSTA,1072,63,✅


---
## 🔍 CELDA 7 — Auditoría de estructura antes de consolidar

In [7]:
if not todos_los_datos:
    raise RuntimeError("❌ No hay datos. Revisa los errores de la Fase 1.")

print("═" * 65)
print("AUDITORÍA DE ESTRUCTURA")
print("═" * 65)

# ── Discrepancias en número de columnas ──────────────────────────────────────
dims = {df['ARCHIVO_ORIGEN'].iloc[0]: df.shape[1] for df in todos_los_datos}
counts_cols = defaultdict(list)
for arch, ncols in dims.items():
    counts_cols[ncols].append(arch)

print(f"\nDistribución de columnas por archivo:")
for ncols, archs in sorted(counts_cols.items(), reverse=True):
    print(f"  {ncols} columnas → {len(archs)} archivo(s)")
    for a in archs:
        print(f"    · {a}")

# ── Diferencias de vocabulario entre archivos ────────────────────────────────
print("\n─── Comparación de columnas entre archivos ───")
cols_ref  = set(todos_los_datos[0].columns)
arch_ref  = todos_los_datos[0]['ARCHIVO_ORIGEN'].iloc[0]

for df in todos_los_datos[1:]:
    arch      = df['ARCHIVO_ORIGEN'].iloc[0]
    cols_este = set(df.columns)
    faltan    = cols_ref - cols_este
    sobran    = cols_este - cols_ref

    if faltan or sobran:
        print(f"\n  📄 {arch}")
        if faltan:
            print(f"     Columnas ausentes vs referencia ({len(faltan)}):")
            for c in sorted(faltan): print(f"       - {c}")
        if sobran:
            print(f"     Columnas extra vs referencia ({len(sobran)}):")
            for c in sorted(sobran): print(f"       + {c}")

print("\n✅ Auditoría completada.")

═════════════════════════════════════════════════════════════════
AUDITORÍA DE ESTRUCTURA
═════════════════════════════════════════════════════════════════

Distribución de columnas por archivo:
  89 columnas → 12 archivo(s)
    · SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx
    · SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx
    · SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx
    · SALA DE DETENIDOS DÍA - 28022025 IT. AYALA.xlsx
    · SALA DE DETENIDOS DÍA - 29092024 IT. AYALA.xlsx
    · SALA DE DETENIDOS DÍA - 31082024 IT. AYALA.xlsx
    · SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx
    · SALA DE DETENIDOS DÍA 30062024 IT. TOBÓN.xlsx
    · SALA DE DETENIDOS DÍA 30092024 SI. GARCIA.xlsx
    · SALA DE DETENIDOS DÍA 30112024 PT. DÁVILA.xlsx
    · SALA DE DETENIDOS DÍA 31032025 PT. DÁVILA.xlsx
    · SALA DE DETENIDOS DÍA 31102024 SI. GARCIA.xlsx
  72 columnas → 1 archivo(s)
    · SALA DE DETENIDOS DÍA 18092024 IT. ACOSTA.xlsx
  70 columnas → 2 archivo(s)
    · SALA DE DETENIDOS 

---
## 🧩 CELDA 8 — FASE 2: Homologación de vocabulario

In [8]:
print("═" * 65)
print("FASE 2 — HOMOLOGACIÓN DE VOCABULARIO")
print("═" * 65)

for i in range(len(todos_los_datos)):
    df = todos_los_datos[i]
    # Aplicar homologación a cada nombre de columna
    nuevos_nombres = [homologar(str(col)) for col in df.columns]
    # Deduplicar tras homologar (dos nombres distintos pueden mapear al mismo)
    todos_los_datos[i].columns = deduplicar(nuevos_nombres)

print("✅ Homologación aplicada a todos los DataFrames.")

# Verificar si las diferencias se redujeron
print("\n─── Verificación post-homologación ───")
todos_cols_sets = [set(df.columns) for df in todos_los_datos]
col_comun = set.intersection(*todos_cols_sets)
col_union  = set.union(*todos_cols_sets)

print(f"  Columnas en TODOS los archivos (inner): {len(col_comun)}")
print(f"  Columnas en ALGÚN archivo (outer)     : {len(col_union)}")
print(f"  Columnas que NO son comunes           : {len(col_union - col_comun)}")

if col_union - col_comun:
    print("\n  Columnas no comunes (aparecen solo en algunos archivos):")
    for c in sorted(col_union - col_comun):
        archs_con_col = [df['ARCHIVO_ORIGEN'].iloc[0]
                         for df in todos_los_datos if c in df.columns]
        print(f"    · '{c}'")
        for a in archs_con_col:
            print(f"      → {a}")

═════════════════════════════════════════════════════════════════
FASE 2 — HOMOLOGACIÓN DE VOCABULARIO
═════════════════════════════════════════════════════════════════
✅ Homologación aplicada a todos los DataFrames.

─── Verificación post-homologación ───
  Columnas en TODOS los archivos (inner): 3
  Columnas en ALGÚN archivo (outer)     : 254
  Columnas que NO son comunes           : 251

  Columnas no comunes (aparecen solo en algunos archivos):
    · '0_0'
      → SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx
      → SALA DE DETENIDOS DÍA 20042025 PT.DÁVILA.xlsx
      → SALA DE DETENIDOS DÍA 27022024 IT. TOBÓN.xlsx
    · '0_0_0'
      → SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx
      → SALA DE DETENIDOS DÍA 20042025 PT.DÁVILA.xlsx
      → SALA DE DETENIDOS DÍA 27022024 IT. TOBÓN.xlsx
    · '0_0_0_1'
      → SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx
      → SALA DE DETENIDOS DÍA 20042025 PT.DÁVILA.xlsx
      → SALA DE DETENIDOS DÍA 27022024 IT. TOBÓN.xlsx
    · '0_0_0_10'

---
## 🚀 CELDA 9 — FASE 3: Consolidación

> **`join='inner'`** → solo columnas presentes en TODOS los archivos.  
> Esto elimina automáticamente las columnas que algunos operadores no registran.  
> Si prefieres mantenerlas como NaN, cambia a `join='outer'`.

In [9]:
print("═" * 65)
print("FASE 3 — CONSOLIDACIÓN")
print("═" * 65)

# ── Concatenar con inner join (solo columnas comunes) ────────────────────────
df_consolidado = pd.concat(
    todos_los_datos,
    axis=0,
    ignore_index=True,
    join='inner'    # ← cambia a 'outer' si quieres mantener todas las columnas
)
print(f"\n📊 Tras concat (inner join): {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

# ── Filtro final: eliminar cualquier fila con TOTAL GENERAL o SUBTOTAL ────────
col_rg     = df_consolidado.columns[3]   # 'RG'  (pos 0-2 son metadatos)
col_unidad = df_consolidado.columns[4]   # 'UNIDAD'

n_antes = len(df_consolidado)

mask_filtrar = (
    df_consolidado[col_unidad].astype(str).str.contains('SUBTOTAL', case=False, na=False) |
    df_consolidado[col_rg].astype(str).str.contains('TOTAL GENERAL', case=False, na=False) |
    df_consolidado[col_unidad].astype(str).str.contains('TOTAL GENERAL', case=False, na=False)
)

df_consolidado = df_consolidado[~mask_filtrar].reset_index(drop=True)
print(f"🧹 Filas de totales/subtotales eliminadas: {n_antes - len(df_consolidado)}")

# ── Eliminar filas donde UNIDAD es nula ──────────────────────────────────────
n_antes = len(df_consolidado)
df_consolidado = df_consolidado.dropna(subset=[col_unidad]).reset_index(drop=True)
print(f"🗑️  Filas sin UNIDAD eliminadas           : {n_antes - len(df_consolidado)}")

print(f"\n✅ Dataset limpio: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

═════════════════════════════════════════════════════════════════
FASE 3 — CONSOLIDACIÓN
═════════════════════════════════════════════════════════════════

📊 Tras concat (inner join): 23,352 filas × 3 columnas


IndexError: index 3 is out of bounds for axis 0 with size 3

---
## 🔢 CELDA 10 — FASE 4: Tipado de columnas

In [ ]:
print("═" * 65)
print("FASE 4 — TIPADO DE COLUMNAS")
print("═" * 65)

# Detectar automáticamente qué columnas son de texto
cols_texto_detectadas = [
    c for c in df_consolidado.columns
    if any(t in str(c).upper() for t in
           ['RG', 'UNIDAD', 'UBICACIÓN', 'ARCHIVO', 'FECHA', 'OPERADOR'])
]

# Unión con las definidas en configuración
cols_texto_final = list(set(list(COLS_TEXTO) + cols_texto_detectadas))
cols_texto_final = [c for c in cols_texto_final if c in df_consolidado.columns]
cols_numericas   = [c for c in df_consolidado.columns if c not in cols_texto_final]

print(f"\nColumnas de texto : {len(cols_texto_final)}")
print(f"Columnas numéricas: {len(cols_numericas)}")

# Convertir columnas numéricas
# Se usa Int64 (pandas nullable integer) en lugar de int64
# para mantener la distinción entre 0 real y dato ausente (NaN)
errores_conversion = []
for col in cols_numericas:
    try:
        df_consolidado[col] = pd.to_numeric(
            df_consolidado[col], errors='coerce'
        ).astype('Int64')
    except Exception as e:
        errores_conversion.append((col, str(e)))

if errores_conversion:
    print("\n⚠️  Columnas con error de conversión:")
    for col, err in errores_conversion:
        print(f"   {col}: {err}")

# Limpiar columnas de texto
for col in cols_texto_final:
    if col in df_consolidado.columns:
        df_consolidado[col] = (
            df_consolidado[col]
            .astype(str)
            .str.strip()
            .replace({'nan': pd.NA, 'None': pd.NA, '': pd.NA})
        )

print("\n✅ Tipado completado.")
print("\nResumen de tipos:")
print(df_consolidado.dtypes.value_counts().to_string())

---
## 📋 CELDA 11 — Vista final y resumen

In [ ]:
print("═" * 65)
print("RESUMEN DEL DATASET CONSOLIDADO FINAL")
print("═" * 65)
print(f"Total de filas          : {df_consolidado.shape[0]:,}")
print(f"Total de columnas       : {df_consolidado.shape[1]}")
print(f"Archivos fuente         : {df_consolidado['ARCHIVO_ORIGEN'].nunique()}")
print(f"Rango de fechas         : {df_consolidado['FECHA_REPORTE'].min()} → {df_consolidado['FECHA_REPORTE'].max()}")
print(f"Operadores únicos       : {df_consolidado['OPERADOR'].nunique()}")

# Columna UNIDAD puede estar en diferentes posiciones según el join
if 'UNIDAD' in df_consolidado.columns:
    print(f"Unidades únicas         : {df_consolidado['UNIDAD'].nunique()}")
if 'UBICACIÓN SALAS' in df_consolidado.columns:
    print(f"Ubicaciones únicas      : {df_consolidado['UBICACIÓN SALAS'].nunique()}")

print("\n─── Registros por archivo ───")
print(df_consolidado['ARCHIVO_ORIGEN'].value_counts().to_string())

print("\n─── Registros por operador ───")
print(df_consolidado['OPERADOR'].value_counts().to_string())

print("\n─── Vista previa (primeras 10 filas, columnas base) ───")
cols_preview = [c for c in df_consolidado.columns[:10]]
display(df_consolidado[cols_preview].head(10))

---
## 🩺 CELDA 12 — Diagnóstico de calidad de datos

In [ ]:
print("═" * 65)
print("DIAGNÓSTICO DE CALIDAD DE DATOS")
print("═" * 65)

total_filas = len(df_consolidado)

# ── NaN por columna ──────────────────────────────────────────────────────────
nan_info = df_consolidado.isnull().sum().reset_index()
nan_info.columns = ['Columna', 'NaN_count']
nan_info['pct_nan'] = (nan_info['NaN_count'] / total_filas * 100).round(1)
nan_info = nan_info[nan_info['NaN_count'] > 0].sort_values('pct_nan', ascending=False)

if nan_info.empty:
    print("\n✅ Sin valores NaN en el dataset.")
else:
    print(f"\n⚠️  {len(nan_info)} columnas con NaN:")
    # Columnas críticas (>50% NaN — posible error de lectura)
    criticas = nan_info[nan_info['pct_nan'] > 50]
    if not criticas.empty:
        print(f"\n  🚨 CRÍTICAS (>50% NaN) — verificar si son datos reales:")
        for _, row in criticas.iterrows():
            print(f"    [{row['pct_nan']:5.1f}%] {row['Columna']}")

    normales = nan_info[nan_info['pct_nan'] <= 50]
    if not normales.empty:
        print(f"\n  ℹ️  Normales (≤50% NaN) — probablemente datos opcionales:")
        for _, row in normales.iterrows():
            print(f"    [{row['pct_nan']:5.1f}%] {row['Columna']}")

# ── Estadísticas de columnas numéricas clave ─────────────────────────────────
print("\n─── Estadísticas de columnas numéricas principales ───")
cols_numericas_presentes = [
    c for c in df_consolidado.columns
    if df_consolidado[c].dtype in ['Int64', 'int64', 'float64']
][:15]  # máximo 15 para no saturar la salida

if cols_numericas_presentes:
    display(
        df_consolidado[cols_numericas_presentes]
        .describe()
        .round(1)
    )

---
## 💾 CELDA 13 — Exportación

In [ ]:
fecha_hora    = datetime.now().strftime('%Y%m%d_%H%M')
nombre_salida = f"sala_detenidos_consolidado_{fecha_hora}.xlsx"
ruta_salida   = os.path.join(RUTA_RESULTADOS, nombre_salida)

print(f"Guardando en: {ruta_salida}")
print("(puede tardar unos segundos con datasets grandes)")

df_consolidado.to_excel(ruta_salida, index=False, engine='openpyxl')

tam_mb = os.path.getsize(ruta_salida) / (1024 * 1024)
print(f"\n✅ Archivo guardado.")
print(f"   Nombre   : {nombre_salida}")
print(f"   Tamaño   : {tam_mb:.2f} MB")
print(f"   Filas    : {df_consolidado.shape[0]:,}")
print(f"   Columnas : {df_consolidado.shape[1]}")
print(f"   Período  : {df_consolidado['FECHA_REPORTE'].min()} → {df_consolidado['FECHA_REPORTE'].max()}")

---
## 🔁 CELDA 14 — Añadir nuevos archivos al dataset existente (uso futuro)

Cuando lleguen nuevos archivos, ejecuta solo esta celda para agregarlos al dataset consolidado sin reprocesar todo.

In [ ]:
# ─── AJUSTAR ESTOS PARÁMETROS ─────────────────────────────────────────────────
RUTA_DATASET_EXISTENTE = ruta_salida          # ← ruta del Excel consolidado ya guardado
ARCHIVOS_NUEVOS        = []                   # ← lista de rutas completas de archivos nuevos
# ─────────────────────────────────────────────────────────────────────────────

if not ARCHIVOS_NUEVOS:
    print("ℹ️  No hay archivos nuevos definidos. Ajusta ARCHIVOS_NUEVOS para usar esta celda.")
else:
    print(f"Cargando dataset existente: {RUTA_DATASET_EXISTENTE}")
    df_existente = pd.read_excel(RUTA_DATASET_EXISTENTE, engine='openpyxl')
    print(f"  Dataset actual: {df_existente.shape[0]:,} filas")

    # Procesar nuevos archivos
    nuevos_dfs = []
    archivos_ya_procesados = set(df_existente['ARCHIVO_ORIGEN'].unique())

    for ruta_nuevo in ARCHIVOS_NUEVOS:
        nombre = os.path.basename(ruta_nuevo)
        if nombre in archivos_ya_procesados:
            print(f"  ⏭️  Ya procesado, omitiendo: {nombre}")
            continue

        print(f"  📖 Procesando: {nombre}")
        df_nuevo = transformar_archivo(ruta_nuevo, verbose=False)
        if df_nuevo is not None:
            nuevos_dfs.append(df_nuevo)
            print(f"     ✅ {df_nuevo.shape[0]:,} filas")

    if nuevos_dfs:
        df_incremental = pd.concat(
            [df_existente] + nuevos_dfs,
            axis=0,
            ignore_index=True,
            join='inner'
        )
        fecha_hora_inc = datetime.now().strftime('%Y%m%d_%H%M')
        ruta_salida_inc = os.path.join(
            RUTA_RESULTADOS,
            f"sala_detenidos_consolidado_{fecha_hora_inc}.xlsx"
        )
        df_incremental.to_excel(ruta_salida_inc, index=False, engine='openpyxl')
        print(f"\n✅ Dataset actualizado guardado: {ruta_salida_inc}")
        print(f"   Filas anteriores: {len(df_existente):,}")
        print(f"   Filas nuevas    : {sum(len(d) for d in nuevos_dfs):,}")
        print(f"   Total final     : {len(df_incremental):,}")
    else:
        print("ℹ️  No se añadieron filas nuevas.")